# 06 — Criteo: the real causal result, and whether the effect is findable

This is the only notebook in ATHAR whose causal claim rests on data rather than on
a construction. Criteo-UPLIFT v2.1 is a genuine randomised trial over 13,979,592
users, 85% of whom were eligible to be shown advertising and 15% held back.

Two questions. First, how far does what a platform reports sit from what actually
happened — computed on the full population, no sampling. Second, given twelve
anonymised features, can a model find *who* the advertising moved.

In [ ]:
import json
import warnings

import pandas as pd

from athar import paths
from athar.provenance import read_metric

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

METRICS = paths.metrics_dir()
PROCESSED = paths.processed_dir()


def show(frame, caption=""):
    if caption:
        print(caption)
    print(frame.to_string(index=False))
    print()

In [ ]:
criteo = read_metric("criteo", METRICS)
print("rows:", f"{criteo['population']['rows']:,}",
      "| treated share:", criteo["population"]["treated_share"])
print("exposure leaking into the control arm:",
      criteo["randomisation_check"]["exposure_in_control_arm"])
print()
print(criteo["randomisation_check"]["note"])

## The intent-to-treat effect

Assignment was random, so the difference between arms is the causal effect of
being eligible for advertising. No adjustment, no model.

In [ ]:
itt = criteo["intent_to_treat"]
conversion = criteo["conversion"]
print(f"treated  {conversion['treated_rate']:.6f}  "
      f"[{conversion['treated_ci'][0]:.6f}, {conversion['treated_ci'][1]:.6f}]")
print(f"control  {conversion['control_rate']:.6f}  "
      f"[{conversion['control_ci'][0]:.6f}, {conversion['control_ci'][1]:.6f}]")
print()
print(f"absolute lift {itt['absolute_lift']:.8f}  95% CI "
      f"[{itt['ci_95'][0]:.8f}, {itt['ci_95'][1]:.8f}]")
print(f"relative lift {itt['relative_lift']:.4f}")
print(f"incremental conversions {itt['incremental_conversions']:,.0f}")

## What a platform reports, against what happened

Both quantities below are correct arithmetic on the same trial. The first
conditions on exposure, which is decided *after* randomisation — who saw an ad
depended on who was browsing, and browsing predicts buying.

Stated as a count, advertising is overstated by a factor of about 1.7. Stated as a
conversion *rate*, by a factor of about 28. Both are reported, because only 3.6% of
the treated arm was ever exposed, and quoting only the dramatic framing would be
rhetoric rather than a finding.

In [ ]:
gap = criteo["platform_reported_versus_incremental"]
naive = gap["naive_rate_framing"]
print(f"platform-reported conversions {gap['platform_reported_conversions']:,}")
print(f"incremental conversions       {gap['incremental_conversions']:,.0f}")
print(f"overstatement                 {gap['overstatement_ratio']}x")
print()
print(f"conversion rate among exposed {naive['conversion_rate_among_exposed']:.6f}")
print(f"conversion rate in control    {naive['conversion_rate_in_control']:.6f}")
print(f"naive rate ratio              {naive['naive_rate_ratio']}x")
print(f"true intent-to-treat ratio    {naive['true_relative_lift_ratio']}x")
print(f"share of treated arm exposed  {naive['exposed_share_of_treated_arm']:.4f}")

## The defensible version of "effect on the exposed"

Random assignment as an instrument for exposure. This recovers the effect on those
the advertising actually reached, without the selection that makes the naive
version meaningless.

In [ ]:
cace = criteo["complier_effect"]
print(f"compliance rate {cace['compliance_rate']:.6f}")
low, high = cace["ci_95"]
print(f"complier effect {cace['cace']:.6f}  95% CI [{low:.6f}, {high:.6f}]")
print()
print(cace["interpretation"])

## Can a model find who was moved

Four learners and a random-targeting baseline, scored by Qini on a held-out half
with a bootstrap interval. Qini subtracts the random-targeting line, so it scores
only what the *ranking* contributed — a model with no uplift signal scores zero
however large the average effect happens to be.

If nothing beats random, that is the finding, and it is a statement about what
twelve anonymised and heavily repeated features support rather than about uplift
modelling.

In [ ]:
try:
    uplift = read_metric("uplift", METRICS)
except FileNotFoundError:
    uplift = None
    print("metrics/uplift.json not present; run `make uplift`.")

if uplift:
    print("test-half control converters:", f"{uplift['sample']['test_control_converters']:,}")
    print()
    rows = [{"model": name, "qini": round(r["qini"], 5),
             "ci_low": round(r["ci_low"], 5), "ci_high": round(r["ci_high"], 5),
             "beats random": r["beats_random"]}
            for name, r in uplift["models"].items()]
    show(pd.DataFrame(rows).sort_values("qini", ascending=False))
    print("verdict:", json.dumps(uplift["verdict"], indent=2))

In [ ]:
if uplift:
    best = uplift["verdict"]["best_model"]
    if best:
        show(pd.DataFrame(uplift["models"][best]["targeting_curve"]).round(4),
             f"Targeting curve — {best}")
        show(pd.DataFrame(uplift["models"][best]["deciles"]).round(6),
             f"Incremental response by decile — {best}")

## Effective sample size

Criteo's twelve features are anonymised and heavily repeated. Where the distinct
share is well below one, the nominal row count overstates the precision available
to a model fitted on those features. It does not affect the treatment-effect
estimates above, which depend on the assignment rather than on the features.

In [ ]:
effective = criteo["effective_sample"]
print(f"distinct feature vectors {effective['distinct_feature_vectors']:,} "
      f"of {criteo['population']['rows']:,} ({effective['distinct_share']:.4f})")
print()
print(effective["note"])